# CSCI 447/547 Hackathon 5: Neural Networks

This notebook is designed to be started during class and continued as a take-home activity

## How to use hackathon notebooks:

If the topic covered in a hackathon is new to you, work through the cells in order and read the explanation before running each code cell. You do not need to understand every detail of the hackathon on the first pass, instead, focus on the approach we take:

1. **Look at the data**
2. **Separate inputs from the target we want to predict**
3. **Split the data so we can test whether the model generalizes**
4. **Fit a model using the training data**
5. **Make predictions and evaluate them**
6. **Improve the model carefully <u>without</u> using the final evaluation data to make decisions**

We will work through the workbook together as a class. Pause before important code cells and think about what you expect to see. Please ask Lucy or a TA questions any time a term or line of code is unfamiliar.

After class, continue from wherever you left off. The existing explanations and code will be there to guide you through the process. Complete the marked answer sections, run every cell, and explain what the results mean in language that makes sense for you. Hackathons will not be graded, they are only to help you, and you will get out of them what you put into them.

<h4><span style="color:red">The goal of this notebook is NOT to memorize every function. It is to identify the processes we use in machine learning and be able to reuse them.</span></h4>

##### In this exercise, you will implement a neural net and train it on a Titanic data set to predict whether an individual survived.

---

### Google Colab Instructions

If you are using Google Colab <a href="https://colab.research.google.com/github/lucywowen/csci547_ML/blob/main/examples/hackathon5-neural_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Google Colab"/></a> you will need to download [ `train_clean.csv` and `test_clean.csv` from GitHub](https://github.com/lucywowen/csci547_ML/tree/main/examples/data/titanic) and upload them to Colab for this to work.

---

### Local (VSCode/Jupyter Lab) Version

To get the latest version of this notebook **without overwriting your work on other hackathons**, open a terminal in your cloned `csci547_ML` folder and run:

```bash
git fetch
git checkout origin/main -- examples/hackathon5-neural_net.ipynb
```

`git fetch` downloads the latest version of the repository without changing any of your files, then `git checkout` replaces only this notebook. *(This overwrites any changes you have made to this notebook, so if you have already started it, make a copy first!)*

Make sure that your `uv` virtual environment is enabled.

This hackathon also uses `tensorflow` (which includes `keras`) to build neural networks. If you don't have it in your environment yet, add it with:

```bash
uv add tensorflow
```

---

## 1. Neural Networks and the Titanic

On April 15, 1912, the RMS Titanic sank after hitting an iceberg, and 1,502 of the 2,224 passengers and crew died. Survival was not random: who you were, where you slept, and who you were travelling with all mattered.

**The Scenario:** Using passenger records, we will train a **neural network** to predict whether a passenger survived (`Survived` = 1) or died (`Survived` = 0). This is a binary classification problem, just like the logistic regression problems we have already seen, and (as we will discover in Section 3) a neural network is built out of many small logistic-regression-like pieces.
### 1.1 Load the Libraries

We will use `keras` to build and train our neural networks. Keras is a high-level library that runs on top of `tensorflow`.

*Note: When you import `keras`, you may see some red log messages from TensorFlow about CUDA, GPUs, or oneDNN. These are harmless. They just mean TensorFlow will train on your CPU, which is plenty fast for this dataset.*

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
sns.set_theme(style='whitegrid')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

import keras
from keras import layers

keras.utils.set_random_seed(42) # makes the random parts of python, numpy, and tensorflow reproducible

### 1.2 Load the Data

The data comes in two files:
* `train_clean.csv`: 891 passengers **with** their `Survived` outcome.
* `test_clean.csv`: 418 passengers **without** a `Survived` outcome. This file comes from a Kaggle competition, so the answers are hidden from us.

(Depending on your environment/data location this cell may need to change.)

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()

    train = pd.read_csv('train_clean.csv') # colab file import
    test = pd.read_csv('test_clean.csv')

except:

    train = pd.read_csv('data/titanic/train_clean.csv') # local file import
    test = pd.read_csv('data/titanic/test_clean.csv')

train.head()

## Data description

| Feature | Description |
| :--- | :--- |
| **Age** | Age in years. |
| **Cabin** | Cabin number (missing for most passengers). |
| **Embarked** | Port the passenger boarded at (C = Cherbourg, Q = Queenstown, S = Southampton). |
| **Fare** | Ticket price. |
| **Name** | Passenger name. |
| **Parch** | Number of parents/children aboard. |
| **PassengerId** | A unique ID number for each passenger. |
| **Pclass** | Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd). |
| **Sex** | male or female. |
| **SibSp** | Number of siblings/spouses aboard. |
| **Ticket** | Ticket number. |
| **Title** | Title taken from the passenger's name (Mr, Mrs, Miss, Master, Dr, Rev). |
| **Family_Size** | `SibSp + Parch`, the number of family members aboard. |
| **Survived** | 0 = died, 1 = survived. *(Target Variable)* |

### Think-Pair-Share 1: Who Survived?
**Context:** We have 13 possible input features for predicting `Survived`.

1. **Think:** Which features do you expect to be the most useful for predicting survival, and why? Are there any features that you think a model should **not** be given? (Look closely at `PassengerId`, `Name`, `Ticket`, and `Cabin`.)

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Discuss your choices with your group.

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

### 1.3 Look at the Data

Let's check your hypotheses. First, what fraction of passengers in the training file survived?

In [ ]:
print(f"Overall survival rate: {train['Survived'].mean():.3f}")

Now let's look at how survival changes across a few of the categorical features.

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(16, 4))

for ax, var in zip(axs, ['Pclass', 'Sex', 'Embarked']):
    sns.countplot(x=var, hue='Survived', data=train, palette='hls', ax=ax)
    ax.set_title(f'Survival by {var}')

plt.show()

## 2. Preparing the Data

Neural networks only understand numbers, and they train best when those numbers are on similar scales. Before we can build our network, we need to **encode** the categorical features and **scale** the continuous features.

### 2.1 Encode Categorical Variables

> We need to convert all categorical variables into numeric format. The categorical variables we will be keeping are `Embarked`, `Sex` and `Title`.

> The `Sex` variable can be encoded into a single 1-or-0 column, but the other variables will need to be **one-hot encoded**. Regular label encoding assigns some category labels higher numerical values. This implies some sort of scale (Embarked = 1 is not more than Embarked = 0, it's just different). One-hot encoding avoids this problem by giving each category its own 0/1 column.

> We will assume that there is some ordinality in the `Pclass` variable (1st > 2nd > 3rd), so we will leave that as a single column.

We combine both files before encoding so that they end up with exactly the same columns. This is safe because encoding doesn't learn anything from the data. Scaling (Section 2.3) is a different story!

In [ ]:
# Stack both files into a single dataframe (test passengers get Survived = NaN)
df = pd.concat([train, test], axis=0, ignore_index=True)

# convert Sex to category dtype, then to category codes (female = 0, male = 1)
df['Sex'] = df['Sex'].astype('category')
df['Sex'] = df['Sex'].cat.codes

# one-hot encode the categorical variables
categorical = ['Embarked', 'Title']

for var in categorical:
    df = pd.concat([df, pd.get_dummies(df[var], prefix=var, dtype=int)], axis=1)
    del df[var]

# drop the variables we won't be using
df = df.drop(columns=['Cabin', 'Name', 'Ticket', 'PassengerId'])

df.head()

### 2.2 Coding Exercise: Split into Train, Fine-Tune, and Generalization Sets

The passengers from `test_clean.csv` have no `Survived` value, so we can't use them to measure how well our model works. Instead, just like in Hackathon 0, we will carve the 891 labeled passengers into three sets:

* **Training (60%):** used to fit the network's weights.
* **Fine-tuning (20%):** used to compare models and choose hyperparameters.
* **Generalization (20%):** touched **once**, at the very end, to estimate how well our final model works on new passengers.

**What you need to write:** Split `X_temp` and `y_temp` into a training set (`X_train`, `y_train`) and a fine-tuning set (`X_fine`, `y_fine`) so that they end up as 60% and 20% of the *full* dataset. Like the first split, it should keep the survival rate the same in both sets (`stratify`), and use `random_state=42` so your numbers match ours.

**What is handled for you:**
* Separating the labeled and unlabeled passengers, and the first split (which holds out the generalization set).

In [ ]:
# Rows with a known Survived value came from train_clean.csv; rows without one came from test_clean.csv
labeled = df[df['Survived'].notna()]
unknown = df[df['Survived'].isna()]

# Separate features (X) and target labels (y). Keras works with float32 numbers.
X = labeled.drop(columns='Survived').astype('float32')
y = labeled['Survived'].astype(int)
X_unknown = unknown.drop(columns='Survived').astype('float32')

# Hold out 20% of the labeled passengers for the final generalization test
X_temp, X_gen, y_temp, y_gen = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Split the remaining 80% into training (60% of the total) and fine-tuning (20% of the total).
# Hint: what fraction of X_temp is 20% of the full dataset?
X_train, X_fine, y_train, y_fine = # --- YOUR CODE HERE ---

print('Training set:      ', X_train.shape)
print('Fine-tuning set:   ', X_fine.shape)
print('Generalization set:', X_gen.shape)
print('Expected shapes: (534, 16), (178, 16), (179, 16)')

### 2.3 Coding Exercise: Scale Continuous Variables

The continuous variables need to be scaled. Remember from the Linear Regression hackathon: when features have very different ranges (`Fare` goes up to 512, `Parch` is usually 0 to 2), gradient descent struggles. Neural networks are trained with gradient descent too, so they have the same problem. We use `StandardScaler` to transform every continuous feature to have mean 0 and standard deviation 1:

$$z = \frac{x-\mu}{\sigma}$$

**Careful!** The mean $\mu$ and standard deviation $\sigma$ must be learned from the **training set only**. If we computed them using the fine-tuning or generalization passengers, information about those passengers would leak into our model, and our evaluation would be overly optimistic. This is the same `fit_transform()` / `transform()` pattern we used with `CountVectorizer` in Hackathon 4.

**What you need to write:** Scale the continuous columns of the training set so that the scaler learns $\mu$ and $\sigma$ from the training data. Then scale the same columns of the fine-tuning set using those *training* values, without letting the scaler learn anything from the fine-tuning data. The scaler's `fit_transform()` and `transform()` methods are the tools for these two jobs.

**What is handled for you:**
* Scaling the generalization and unknown sets (the same pattern as the fine-tuning set).

In [ ]:
continuous = ['Age', 'Fare', 'Parch', 'Pclass', 'SibSp', 'Family_Size']

# Make independent copies so we can safely overwrite columns
X_train, X_fine, X_gen, X_unknown = X_train.copy(), X_fine.copy(), X_gen.copy(), X_unknown.copy()

scaler = StandardScaler()

# 1. Learn the mean and standard deviation from the TRAINING data, and scale the training data.
X_train[continuous] = # --- YOUR CODE HERE ---

# 2. Scale the fine-tuning data using the training mean and standard deviation (no re-fitting!).
X_fine[continuous] = # --- YOUR CODE HERE ---

# 3. (Handled for you) The same transformation for the remaining sets
X_gen[continuous] = scaler.transform(X_gen[continuous])
X_unknown[continuous] = scaler.transform(X_unknown[continuous])

X_train[continuous].describe().round(2)

Check your work: every continuous column in the training set should now have a mean of 0.00 and a standard deviation of 1.00. (Why won't the fine-tuning set be *exactly* 0 and 1?)

## 3. Neural Networks Under the Hood

Before we let `keras` do the work, let's see what a neural network actually computes.

### 3.1 One Neuron = Logistic Regression

Recall logistic regression: take a weighted sum of the inputs, then squash it into a probability with the sigmoid function. A single **neuron** does exactly the same thing:

$$a = g(\mathbf{w}^T\mathbf{x} + b)$$

* $\mathbf{x}$ is the input (one passenger's 16 features).
* $\mathbf{w}$ are the **weights** and $b$ is the **bias** (the same thing as $\theta_0$ in logistic regression).
* $g$ is the **activation function**. Two common choices:
    * **Sigmoid:** $\sigma(z) = \dfrac{1}{1+e^{-z}}$ squashes any number into $(0, 1)$, so we use it on the output neuron to get a probability.
    * **ReLU:** $\text{ReLU}(z) = \max(0, z)$ is the most common choice for hidden layers. It is cheap to compute and works well in practice.

### 3.2 Many Neurons = A Layer

A **dense layer** is a group of neurons that all look at the same inputs, but each neuron has its own weights. If we stack each neuron's weight vector as a column of a matrix $W$, we can compute the output of the whole layer for all $m$ passengers at once:

$$A = g(XW + \mathbf{b})$$

| Variable | Shape | Meaning |
| :--- | :--- | :--- |
| $X$ | $(m, n_{in})$ | $m$ passengers, $n_{in}$ inputs each |
| $W$ | $(n_{in}, n_{out})$ | one column of weights for each of the $n_{out}$ neurons |
| $\mathbf{b}$ | $(n_{out},)$ | one bias for each neuron |
| $A$ | $(m, n_{out})$ | each neuron's output for each passenger |

### 3.3 Many Layers = A Neural Network

A **neural network** feeds the output of one layer into the next. The layers between the input and the output are called **hidden layers**:

$$A^{[1]} = \text{ReLU}(XW^{[1]} + \mathbf{b}^{[1]}) \qquad \text{(hidden layer)}$$
$$\hat{y} = \sigma(A^{[1]}W^{[2]} + \mathbf{b}^{[2]}) \qquad \text{(output layer)}$$

The output layer is just logistic regression, but instead of using our raw features, it uses features *learned* by the hidden layer. In Hackathon 3 and the Logistic Regression bonus hackathon, we hand-crafted new polynomial features. A neural network learns its own.

### 3.4 Coding Exercise: A Dense Layer in NumPy

**What you need to write:** Finish `dense_forward()` so that it returns the layer's output $A = g(XW + \mathbf{b})$ for every example at once: the weighted sum of the inputs plus the bias, passed through whichever activation function $g$ it was given. (In NumPy, `@` is matrix multiplication.)

**What is handled for you:**
* The `sigmoid` and `relu` activation functions, and a test with a known answer.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

def dense_forward(A_prev, W, b, activation):
    """
    Computes the output of one dense (fully-connected) layer.
    A_prev: inputs to the layer, shape (m, n_in)
    W: weight matrix, shape (n_in, n_out)
    b: bias vector, shape (n_out,)
    activation: the activation function to apply (e.g. sigmoid or relu)
    """
    # 1. Compute the weighted sum for every example at once.
    Z = # --- YOUR CODE HERE ---

    # 2. Apply the activation function.
    A = # --- YOUR CODE HERE ---

    return A


# --- Test your function on a single neuron with two inputs: ---
# z = (1)(0.5) + (2)(-0.25) + 0.1 = 0.1, and sigmoid(0.1) = 0.525
A_test = np.array([[1.0, 2.0]])
W_test = np.array([[0.5], [-0.25]])
b_test = np.array([0.1])

print('Neuron output:', dense_forward(A_test, W_test, b_test, sigmoid))
print('Expected output (approx): [[0.525]]')

### 3.5 Coding Exercise: Stacking Layers into a Network

Now let's chain two layers together to build a tiny network: 16 inputs → 8 hidden neurons (ReLU) → 1 output neuron (sigmoid). We'll fill the weights with random numbers and run the first 5 training passengers through it.

**What you need to write:** Compute the network's `output`. The output layer takes the **hidden layer's output** as its input, has the weights `W2` and biases `b2`, and uses a sigmoid activation so that every passenger gets a survival probability.

In [ ]:
rng = np.random.default_rng(42)
n_features = X_train.shape[1]

# Random (untrained!) weights: 16 inputs -> 8 hidden neurons -> 1 output
W1, b1 = rng.normal(size=(n_features, 8)), np.zeros(8)
W2, b2 = rng.normal(size=(8, 1)), np.zeros(1)

X_first5 = X_train.to_numpy()[:5]

# Hidden layer: ReLU activation
hidden = dense_forward(X_first5, W1, b1, relu)

# Output layer: sigmoid activation, using the hidden layer's output as its input
output = # --- YOUR CODE HERE ---

print('Hidden layer output shape:', hidden.shape, '  Expected: (5, 8)')
print('Network output shape:     ', output.shape, '  Expected: (5, 1)')
print()
print('Predicted survival probabilities:', output.ravel().round(3))
print('Actual outcomes:                 ', y_train.to_numpy()[:5])

Our network makes predictions, but they're meaningless: the weights are random! **Training** a neural network means finding the values for all of the numbers in `W1`, `b1`, `W2`, and `b2` that make the predictions match the actual outcomes. That's what we'll use `keras` for in Section 4.

### Think-Pair-Share 2: Why Do We Need Non-Linear Activations?
**Context:** Suppose we built our network with a *linear* activation in the hidden layer, meaning the hidden layer applies no activation function at all: $A^{[1]} = XW^{[1]} + \mathbf{b}^{[1]}$.

1. **Think:** Substitute this into the output layer, $\hat{y} = \sigma(A^{[1]}W^{[2]} + \mathbf{b}^{[2]})$, and simplify. What kind of model do you end up with? Would adding more linear hidden layers help?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Discuss with your group. What does the "kink" in ReLU at $z = 0$ let a network do that a stack of linear layers can't?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

Two linear layers multiply out into a single linear layer:

$$\hat{y} = \sigma\big((XW^{[1]} + \mathbf{b}^{[1]})W^{[2]} + \mathbf{b}^{[2]}\big) = \sigma\big(X\underbrace{W^{[1]}W^{[2]}}_{W} + \underbrace{\mathbf{b}^{[1]}W^{[2]} + \mathbf{b}^{[2]}}_{\mathbf{b}}\big)$$

which is exactly logistic regression. Let's check this numerically with the weights from above:

In [ ]:
def identity(z):
    return z # a "linear" activation that does nothing

two_linear_layers = dense_forward(dense_forward(X_first5, W1, b1, identity), W2, b2, sigmoid)
logistic_regression = dense_forward(X_first5, W1 @ W2, b1 @ W2 + b2, sigmoid)

print('Same predictions?', np.allclose(two_linear_layers, logistic_regression))

## 4. Building and Training a Network with Keras

### 4.1 Coding Exercise: Create the Model

> For this task, we will keep the model architecture pretty simple. We have an input layer with 16 inputs (one per feature) which feeds into a hidden layer with 8 neurons, and an output layer with a single neuron which is used to predict a passenger's survival.

> The output layer has a sigmoid activation function, which is used to 'squash' all our outputs to be between 0 and 1.

> We are going to create a function which allows us to choose the hyperparameters of the neural network. This might seem a little overly complicated now, but it will come in super handy when we move onto tuning our parameters later.

In Keras, a `Sequential` model is a stack of layers, and `layers.Dense(n_neurons, activation=...)` is exactly the dense layer you just wrote in NumPy. Keras creates and stores the `W` and `b` for each layer for us.

**What you need to write:** Finish `create_model()` so that every hidden layer is a `layers.Dense()` layer with `n_neurons` neurons that uses the activation function `act`, and so that the network ends in an output layer with a single neuron whose sigmoid activation turns its output into a survival probability.

**What is handled for you:**
* The input layer, dropout (more on this in Section 5), and choosing the loss function, optimizer, and metric with `model.compile()`.

In [ ]:
def create_model(lyrs=[8], act='relu', opt='adam', dr=0.0):
    """
    Builds and compiles a fully-connected neural network for binary classification.
    lyrs: number of neurons in each hidden layer, e.g. [32, 16] means two hidden layers
    act: activation function for the hidden layers
    opt: optimization algorithm used during training
    dr: dropout rate (the fraction of neurons randomly switched off during training)
    """
    # set random seed for reproducibility
    keras.utils.set_random_seed(42)

    model = keras.Sequential()

    # input layer: one input per feature
    model.add(keras.Input(shape=(X_train.shape[1],)))

    # create the hidden layers
    for n_neurons in lyrs:
        # 1. A Dense layer with n_neurons neurons and the activation function act
        hidden_layer = # --- YOUR CODE HERE ---
        model.add(hidden_layer)

    # add dropout, default is none
    model.add(layers.Dropout(dr))

    # 2. The output layer: a single neuron with a sigmoid activation
    output_layer = # --- YOUR CODE HERE ---
    model.add(output_layer)

    # (Handled for you) choose the loss function, optimizer, and the metric to report
    model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])

    return model

**Pause point:** Our default network has 16 inputs, one hidden layer of 8 neurons, and 1 output neuron. Before running the next cell, calculate by hand how many **trainable parameters** (weights + biases) the network has. Hint: what are the shapes of `W1`, `b1`, `W2`, and `b2` from Section 3.5?

In [ ]:
model = create_model()
model.summary()
print('Expected total params: 145')

### 4.2 Coding Exercise: Train the Model

> At this stage, we have our model. We have chosen a few hyperparameters such as the number of hidden layers, the number of neurons and the activation function.

> The next step is to train the model on our training set. This step also requires us to choose a few more hyperparameters such as the loss function, the optimization algorithm, the number of epochs and the batch size.

* **Loss function:** We use **binary cross-entropy**, which is the same log loss we minimized for logistic regression:
$$J = -\frac{1}{m}\sum_{i=1}^m\left[y^{(i)}\log(\hat{y}^{(i)}) + (1-y^{(i)})\log(1-\hat{y}^{(i)})\right]$$
* **Backpropagation:** Uses the chain rule from calculus to compute the gradient of $J$ with respect to *every* weight in *every* layer, working backwards from the output layer.
* **Optimizer:** **Adam** is a version of gradient descent that automatically adapts the learning rate $\alpha$ for each weight.
* **Epoch:** One full pass through the training data.
* **Batch size:** How many passengers are used to compute each gradient descent step. With 534 training passengers and a batch size of 32, each epoch takes $\lceil 534/32 \rceil = 17$ steps.
* **Validation data:** After every epoch, Keras reports the loss and accuracy on the fine-tuning set. This data is only used for *reporting*, never for updating the weights.

**What you need to write:** Train `model` on the training set for 100 epochs with a batch size of 32, and have Keras report the loss and accuracy on the fine-tuning set after every epoch (Keras calls this `validation_data`). The model's `fit()` method returns the training history that the next cells plot. Set `verbose=0`, or Keras will print 100 progress bars. Training takes a few seconds.

In [ ]:
history = # --- YOUR CODE HERE ---

print(f"Final training accuracy:    {history.history['accuracy'][-1]:.3f}")
print(f"Final fine-tuning accuracy: {history.history['val_accuracy'][-1]:.3f}")
print('Expected (approx): 0.84 and 0.85 (your exact numbers may differ slightly)')

### 4.3 Learning Curves

`history.history` is a dictionary that stores the loss and accuracy on the training and fine-tuning sets after every epoch. Plotting these **learning curves** is the best way to see how training went.

In [ ]:
def plot_history(history, title):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))
    for ax, metric in zip(axs, ['loss', 'accuracy']):
        ax.plot(history.history[metric], label='train')
        ax.plot(history.history['val_' + metric], label='fine-tune (validation)')
        ax.set(xlabel='epoch', ylabel=metric, title=f'{title}: {metric}')
        ax.legend()
    plt.show()

plot_history(history, 'Small network')

### 4.4 Coding Exercise: What Overfitting Looks Like

Our small network has 145 parameters to fit 534 training passengers. What happens if we use a much bigger network and train it for longer?

A network with two hidden layers of 128 neurons each has $(16 \times 128 + 128) + (128 \times 128 + 128) + (128 + 1) = 18,817$ parameters. That's about 35 parameters for every training passenger!

**What you need to write:** Use `create_model()` to build `big_model`, a network with **two hidden layers of 128 neurons each**.

In [ ]:
big_model = # --- YOUR CODE HERE ---

big_history = big_model.fit(X_train, y_train, validation_data=(X_fine, y_fine), epochs=200, batch_size=32, verbose=0)
plot_history(big_history, 'Big network')

### Think-Pair-Share 3: Reading Learning Curves
**Context:** Compare the learning curves of the big network to those of the small network in Section 4.3.

1. **Think:** Look at the big network's fine-tuning (validation) **loss**. At roughly what epoch is it lowest, and what happens after that? Meanwhile, what is the training loss doing? What is the network doing to the training data?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Discuss with your group. List at least two changes that could fix this problem. (Hint: look at the arguments of `create_model()` and `model.fit()`.)

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

## 5. Improving the Model: Hyperparameter Tuning

We've made a lot of choices: the number of hidden layers, the number of neurons in each layer, the activation function, the number of epochs, and more. These are all **hyperparameters**: settings that we choose, not weights that the network learns.

The original Kaggle notebook tuned these with `GridSearchCV` (like in Hackathon 0). Training neural networks is slow, so instead we'll try a handful of configurations ourselves. Each configuration is trained on the **training set** and scored on the **fine-tuning set**. Following step 6 of our workflow, we will **not** look at the generalization set while making these decisions.

One of the hyperparameters we will try is **dropout**. During each training step, dropout randomly switches off a fraction `dr` of the neurons, so the network can't rely on any one neuron to memorize the training data. Like ridge and lasso, dropout is a form of **regularization**. Dropout is automatically turned off when we make predictions.

### 5.1 Coding Exercise: Score Each Configuration

**What you need to write:** Measure each trained model's loss and accuracy on the fine-tuning set. A Keras model's `evaluate()` method takes features and labels and returns `[loss, accuracy]` (`verbose=0` keeps it quiet).

**What is handled for you:**
* Building and training each model, and collecting the results into a table. This cell takes about 30 seconds.

In [ ]:
configs = [
    {'lyrs': [8],      'act': 'linear', 'dr': 0.0},
    {'lyrs': [8],      'act': 'relu',   'dr': 0.0},
    {'lyrs': [32, 16], 'act': 'relu',   'dr': 0.0},
    {'lyrs': [32, 16], 'act': 'relu',   'dr': 0.3},
    {'lyrs': [64, 64], 'act': 'relu',   'dr': 0.5},
]

results = []
for cfg in configs:
    model = create_model(lyrs=cfg['lyrs'], act=cfg['act'], dr=cfg['dr'])
    model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=0)

    # Evaluate the trained model on the FINE-TUNING set
    fine_loss, fine_acc = # --- YOUR CODE HERE ---

    results.append({**cfg, 'fine_loss': fine_loss, 'fine_acc': fine_acc}) # copy the settings and add the scores
    print(cfg, f'fine-tuning accuracy: {fine_acc:.3f}')

results_df = pd.DataFrame(results).sort_values('fine_acc', ascending=False)
results_df

**Pause point:** The fine-tuning set has 178 passengers, so each passenger is worth $1/178 \approx 0.56\%$ accuracy. How many passengers separate the best configuration from the worst? How much would you trust that difference? Does the `linear` network do worse than the `relu` networks, as Think-Pair-Share 2 predicted?

### 5.2 Train the Final Model

Now that we've made our choices, we retrain the best configuration on the training **and** fine-tuning data combined (more data = a better model), just like we did in Hackathon 0.

In [ ]:
best = results_df.iloc[0]

X_combined = pd.concat([X_train, X_fine])
y_combined = pd.concat([y_train, y_fine])

final_model = create_model(lyrs=best['lyrs'], act=best['act'], dr=best['dr'])
final_model.fit(X_combined, y_combined, epochs=100, batch_size=32, verbose=0)

print('Final model trained with the best configuration:', best['lyrs'], best['act'], best['dr'])

## 6. Evaluate Generalization

Now, and only now, we use the generalization set to estimate how well our final model will work on passengers it has never seen.

### 6.1 Coding Exercise: From Probabilities to Predictions

`final_model.predict()` returns the output of the sigmoid neuron: a survival **probability** for each passenger, with shape `(179, 1)`. To get a prediction, we say a passenger survived if their probability is above 0.5.

**What you need to write:** Turn the probabilities in `gen_probs` into `gen_pred`: a flat (1D) array of predictions that is 1 for every passenger whose survival probability is above 0.5, and 0 for everyone else. Hint: comparing a NumPy array to a number (like `gen_probs > 0.5`) gives an array of `True`/`False` values, `.astype(int)` turns those into 1s and 0s, and `.ravel()` flattens an array to 1D.

In [ ]:
gen_probs = final_model.predict(X_gen, verbose=0)

gen_pred = # --- YOUR CODE HERE ---

gen_acc = accuracy_score(y_gen, gen_pred)
print(f'Generalization accuracy: {gen_acc:.3f}')

ConfusionMatrixDisplay.from_predictions(y_gen, gen_pred, display_labels=['Died', 'Survived'])
plt.title('Neural Network Confusion Matrix (Generalization Set)')
plt.grid(False)
plt.show()

**Pause point:** Is this a good accuracy? Compare it to a "model" that does no learning at all and simply predicts that **every** passenger died.

In [ ]:
baseline_acc = (y_gen == 0).mean()
print(f"'Everyone died' baseline accuracy: {baseline_acc:.3f}")

### 6.2 Compare with Logistic Regression

In Section 3, we saw that a neural network is built from logistic-regression-like neurons. So how does plain logistic regression do when trained and tested on exactly the same data?

In [ ]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_combined, y_combined)

print(f'Logistic regression generalization accuracy: {log_reg.score(X_gen, y_gen):.3f}')
print(f'Neural network generalization accuracy:      {gen_acc:.3f}')

### 6.3 Predict the Unknown Passengers

Finally, the 418 passengers from `test_clean.csv`. We can't score these predictions since their outcomes are hidden, but this is what a model is ultimately for: making predictions on new data.

In [ ]:
unknown_pred = (final_model.predict(X_unknown, verbose=0) > 0.5).astype(int).ravel()
print(f'Predicted survivors: {unknown_pred.sum()} of {len(unknown_pred)} passengers')

test[['Name', 'Sex', 'Age', 'Pclass']].assign(Predicted_Survived=unknown_pred).head(10)

---

## Hackathon Takeaways & Synthesis

As your group finishes this notebook, collaborate to answer the following synthesis questions.

1. **Trade-off Analysis:**
Look at your results in Section 6.2. Did the neural network beat plain logistic regression? A neural network has more parameters, more hyperparameters to tune, and takes longer to train than logistic regression. Why might that extra flexibility not help on a dataset of ~900 passengers with 16 features? When *is* it worth it?

    **ANSWER**

2. **Diagnosing Failures:**
A classmate trains a network and reports 97% training accuracy but only 76% fine-tuning accuracy, and their fine-tuning loss has risen every epoch since epoch 15. What is the problem? Name two specific changes to `create_model()` or `model.fit()` that could fix it.

    **ANSWER**

3. **3-2-1 Summary:**
* **3** key mathematical or programming concepts your group solidified today:

    1. 

    2. 

    3. 

* **2** things you are still slightly confused about (we will address these in the next lecture!):

    1. 

    2. 

* **1** real-world problem where you would choose a neural network over logistic regression:

    1. 